In [2]:
!pip install requests

In [3]:
import requests, json, time, pandas as pd
from tqdm import tqdm

base = "https://www.dnd5eapi.co/api"
slugs = requests.get(f"{base}/spells").json()["results"]
print(f"Total spells in SRD: {len(slugs)}")

spells = []
for s in tqdm(slugs):
    data = requests.get(f"{base}/spells/{s['index']}").json()
    spells.append({
        "name":           data["name"],
        "desc":           " ".join(data["desc"]),
        "level":          data["level"],
        "school":         data["school"]["name"],
        "classes":        [c["name"] for c in data["classes"]],
        "cast_time":      data["casting_time"],
        "range":          data["range"],
        "duration":       data["duration"],
        "verbal":         "V" in data["components"],
        "somatic":        "S" in data["components"],
        "material":       "M" in data["components"],
        "material_desc":  data.get("material", ""),
        "higher_levels":  " ".join(data.get("higher_level", [])),
        "concentration":  data.get("concentration", False),
        "ritual":         data.get("ritual", False),
        "attack_type":    data.get("attack_type", ""),
        "damage_type":    data.get("damage", {}).get("damage_type", {}).get("name", ""),
        "subclasses":     [c["name"] for c in data.get("subclasses", [])],
        "dc_type":        data.get("dc", {}).get("dc_type", {}).get("name", ""),
    })
    time.sleep(0.05)

df = pd.DataFrame(spells)
df.to_csv("spells_full.csv", index=False)
print("Done:", df.shape)

Total spells in SRD: 319


100%|██████████| 319/319 [01:21<00:00,  3.93it/s]

Done: (319, 19)


In [8]:

!wget -q "https://raw.githubusercontent.com/5etools-mirror-3/5etools-src/main/data/spells/spells-phb.json" -O spells_phb.json
!wget -q "https://raw.githubusercontent.com/5etools-mirror-3/5etools-src/main/data/spells/spells-xge.json" -O spells_xge.json
!wget -q "https://raw.githubusercontent.com/5etools-mirror-3/5etools-src/main/data/spells/spells-tce.json" -O spells_tce.json
!wget -q "https://raw.githubusercontent.com/5etools-mirror-3/5etools-src/main/data/spells/spells-ftd.json" -O spells_ftd.json

import json, glob, pandas as pd

def parse_5etools_entry(entry):
    """Recursively extract text from 5etools nested entry format."""
    if isinstance(entry, str): return entry
    if isinstance(entry, dict):
        t = entry.get("type", "")
        if t in ("entries", "list"):
            return " ".join(parse_5etools_entry(e) for e in entry.get("entries", entry.get("items", [])))
        if t == "table":
            return ""  # skip tables
        return parse_5etools_entry(entry.get("entries", entry.get("entry", "")))
    if isinstance(entry, list):
        return " ".join(parse_5etools_entry(e) for e in entry)
    return ""

rows = []
for fpath in glob.glob("spells_*.json"):
    data = json.load(open(fpath))
    for s in data.get("spell", []):
        desc = " ".join(parse_5etools_entry(e) for e in s.get("entries", []))
        higher = " ".join(parse_5etools_entry(e) for e in s.get("entriesHigherLevel", []))
        classes = [c["name"] for c in s.get("classes", {}).get("fromClassList", [])]
        comp = s.get("components", {})
        school_map = {"A":"Abjuration","C":"Conjuration","D":"Divination",
                      "E":"Enchantment","V":"Evocation","I":"Illusion",
                      "N":"Necromancy","T":"Transmutation"}
        rows.append({
            "name":         s.get("name", ""),
            "desc":         desc,
            "higher_levels":higher,
            "level":        s.get("level", 0),
            "school":       school_map.get(s.get("school", ""), s.get("school", "")),
            "classes":      classes,
            "cast_time":    str(s.get("time", [{}])[0].get("number", "")) + " " + str(s.get("time", [{}])[0].get("unit", "")),
            "range":        str(s.get("range", {}).get("distance", {}).get("amount", "")) + " " + str(s.get("range", {}).get("distance", {}).get("type", "")),
            "duration":     str(s.get("duration", [{}])[0].get("duration", {}).get("amount", "")) + " " + str(s.get("duration", [{}])[0].get("type", "")),
            "verbal":       comp.get("v", False),
            "somatic":      comp.get("s", False),
            "material":     bool(comp.get("m", False)),
            "material_desc":comp.get("m", "") if isinstance(comp.get("m"), str) else "",
            "concentration":any(d.get("concentration", False) for d in s.get("duration", [])),
            "ritual":       s.get("meta", {}).get("ritual", False),
            "damage_type":  "",
            "dc_type":      "",
            "attack_type":  "",
        })

df_5e = pd.DataFrame(rows)
df_5e = df_5e[df_5e["name"].str.strip() != ""]
df_5e = df_5e[df_5e["desc"].str.strip() != ""]
df_5e.to_csv("spells_5etools.csv", index=False)
print(df_5e.shape)

(484, 18)


In [10]:
import os
import json
import glob
import ast
import re

import requests
import pandas as pd
from tqdm import tqdm

OUT_CSV = "spells_master.csv"
ORIG_CSV = "/content/spells_full.csv"

# spell school names
SCHOOL_MAP = {
    "A": "Abjuration",
    "C": "Conjuration",
    "D": "Divination",
    "E": "Enchantment",
    "V": "Evocation",
    "I": "Illusion",
    "N": "Necromancy",
    "T": "Transmutation",
}

# clean text
def clean_str(s):
    if not isinstance(s, str):
        return ""

    s = re.sub(r"\{@\w+ ([^}]+)\}", r"\1", s)
    s = re.sub(r"\{@\w+\}", "", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s

# parse list columns
def parse_list_col(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    try:
        return ast.literal_eval(x)
    except:
        return [str(x)]

# convert values to bool
def normalise_bool(v):
    if isinstance(v, bool):
        return v

    if isinstance(v, str):
        return v.strip().lower() in ("true", "yes", "1")

    return bool(v)

print("Loading 5etools data")

BASE_URL = "https://raw.githubusercontent.com/5etools-mirror-3/5etools-src/main/data/spells"

BOOKS = [
    "spells-phb",
    "spells-xge",
    "spells-tce",
    "spells-ftd",
    "spells-srd",
    "spells-ai",
    "spells-egw",
    "spells-erlw",
    "spells-ggr",
    "spells-idrotf",
    "spells-llok",
    "spells-scc",
    "spells-aag",
    "spells-bmt",
    "spells-dodk",
    "spells-ghloe",
]

# download json files
for book in tqdm(BOOKS):
    fname = f"{book}.json"

    if not os.path.exists(fname):
        try:
            r = requests.get(f"{BASE_URL}/{fname}", timeout=15)

            if r.status_code == 200:
                with open(fname, "w") as f:
                    f.write(r.text)

            else:
                print(f"Skipping {book}")

        except Exception as e:
            print(f"Failed {book}: {e}")

# flatten nested entries
def parse_entry(e):
    if isinstance(e, str):
        return clean_str(e)

    if isinstance(e, list):
        return " ".join(parse_entry(x) for x in e)

    if isinstance(e, dict):
        t = e.get("type", "")

        if t in ("entries", "inset", "insetReadaloud"):
            return " ".join(parse_entry(x) for x in e.get("entries", []))

        if t == "list":
            return " ".join(parse_entry(x) for x in e.get("items", []))

        if t == "item":
            name_part = e.get("name", "")
            entry_part = parse_entry(e.get("entry", e.get("entries", "")))

            return f"{name_part}: {entry_part}".strip(": ")

        if t in ("table", "image"):
            return ""

        for key in ("entries", "entry", "items"):
            if key in e:
                return parse_entry(e[key])

    return ""

# parse spell range
def parse_range(r):
    if not isinstance(r, dict):
        return ""

    rt = r.get("type", "")

    if rt == "special":
        return r.get("text", "special")

    if rt == "point":
        dist = r.get("distance", {})

        amt = dist.get("amount", "")
        typ = dist.get("type", "")

        if amt:
            return f"{amt} {typ}".strip()

        return typ

    return rt

# parse duration
def parse_duration(dur_list):
    if not dur_list:
        return ""

    d = dur_list[0]

    dt = d.get("type", "")

    if dt == "instant":
        return "Instantaneous"

    if dt == "special":
        return "Special"

    if dt == "permanent":
        return "Until dispelled"

    inner = d.get("duration", {})

    amt = inner.get("amount", "")
    unit = inner.get("type", "")

    conc = d.get("concentration", False)

    s = f"{amt} {unit}".strip() if amt else unit

    if conc:
        return f"Concentration, up to {s}"

    return s

# parse casting time
def parse_cast_time(time_list):
    if not time_list:
        return ""

    t = time_list[0]

    num = t.get("number", "")
    unit = t.get("unit", "")

    return f"{num} {unit}".strip() if num else unit

rows_5e = []

# read all spell files
for fpath in sorted(glob.glob("spells-*.json")):
    try:
        data = json.load(open(fpath, encoding="utf-8"))

    except Exception as e:
        print(f"Error reading {fpath}")
        continue

    for s in data.get("spell", []):

        desc = parse_entry(s.get("entries", []))

        higher = parse_entry(s.get("entriesHigherLevel", []))

        comp = s.get("components", {})

        classes = [
            c["name"]
            for c in s.get("classes", {}).get("fromClassList", [])
        ]

        subclasses = [
            c.get("subclass", {}).get("name", "")
            for c in s.get("classes", {}).get("fromSubclass", [])
            if c.get("subclass", {}).get("name", "")
        ]

        damage_type = ""

        if "damageInflict" in s and s["damageInflict"]:
            damage_type = s["damageInflict"][0]

        dc_type = ""

        if "savingThrow" in s and s["savingThrow"]:
            dc_type = s["savingThrow"][0].upper()

        attack_type = ""

        if "spellAttack" in s:
            mapping = {
                "R": "ranged",
                "M": "melee"
            }

            attack_type = mapping.get(s["spellAttack"][0], "")

        rows_5e.append({
            "name": clean_str(s.get("name", "")),
            "desc": desc,
            "higher_levels": higher,
            "level": int(s.get("level", 0)),
            "school": SCHOOL_MAP.get(s.get("school", ""), s.get("school", "")),
            "classes": classes,
            "subclasses": subclasses,
            "cast_time": parse_cast_time(s.get("time", [])),
            "range": parse_range(s.get("range", {})),
            "duration": parse_duration(s.get("duration", [])),
            "verbal": bool(comp.get("v", False)),
            "somatic": bool(comp.get("s", False)),
            "material": bool(comp.get("m", False)),
            "material_desc": comp.get("m", "") if isinstance(comp.get("m"), str) else "",
            "concentration": any(
                d.get("concentration", False)
                for d in s.get("duration", [])
            ),
            "ritual": bool(s.get("meta", {}).get("ritual", False)),
            "damage_type": damage_type.title(),
            "dc_type": dc_type,
            "attack_type": attack_type,
            "source": s.get("source", "5etools"),
        })

df_5e = pd.DataFrame(rows_5e)

print(f"5etools rows: {len(df_5e)}")

print("Loading Open5e data")

open5e_spells = []

url = "https://api.open5e.com/v1/spells/?limit=100&format=json"

# fetch api pages
while url:
    try:
        r = requests.get(url, timeout=15).json()

        open5e_spells.extend(r["results"])

        url = r.get("next")

    except Exception as e:
        print(f"Open5e error: {e}")
        break

print(f"Open5e rows: {len(open5e_spells)}")

# split class string
def open5e_classes(s):
    raw = s.get("dnd_class", "")

    if not raw:
        return []

    return [c.strip() for c in raw.split(",") if c.strip()]

rows_o5 = []

# build open5e rows
for s in open5e_spells:
    rows_o5.append({
        "name": clean_str(s.get("name", "")),
        "desc": clean_str(s.get("desc", "")),
        "higher_levels": clean_str(s.get("higher_level", "")),
        "level": int(s.get("level_int", s.get("level", 0)) or 0),
        "school": s.get("school", "").title(),
        "classes": open5e_classes(s),
        "subclasses": [],
        "cast_time": s.get("casting_time", ""),
        "range": s.get("range", ""),
        "duration": s.get("duration", ""),
        "verbal": "V" in s.get("components", ""),
        "somatic": "S" in s.get("components", ""),
        "material": "M" in s.get("components", ""),
        "material_desc": clean_str(s.get("material", "")),
        "concentration": str(s.get("concentration", "")).strip().lower() == "yes",
        "ritual": str(s.get("ritual", "")).strip().lower() == "yes",
        "damage_type": "",
        "dc_type": "",
        "attack_type": "",
        "source": s.get("document__slug", "open5e"),
    })

df_o5 = pd.DataFrame(rows_o5)

print(f"Parsed Open5e rows: {len(df_o5)}")

print("Loading original CSV")

df_orig = None

if ORIG_CSV and os.path.exists(ORIG_CSV):
    df_orig = pd.read_csv(ORIG_CSV)

    for col in ["classes", "subclasses"]:
        if col in df_orig.columns:
            df_orig[col] = df_orig[col].apply(parse_list_col)

    if "source" not in df_orig.columns:
        df_orig["source"] = "original"

    print(f"Original rows: {len(df_orig)}")

else:
    print("Original CSV not found")

frames = [df_5e, df_o5]

# merge original csv
if df_orig is not None:

    for col in df_5e.columns:
        if col not in df_orig.columns:

            if col in [
                "verbal",
                "somatic",
                "material",
                "concentration",
                "ritual"
            ]:
                df_orig[col] = False

            else:
                df_orig[col] = ""

    frames.append(df_orig[df_5e.columns])

df_all = pd.concat(frames, ignore_index=True)

print(f"Rows before dedup: {len(df_all)}")

# fix bool columns
for col in [
    "verbal",
    "somatic",
    "material",
    "concentration",
    "ritual"
]:
    df_all[col] = df_all[col].apply(normalise_bool)

# clean string columns
for col in [
    "name",
    "desc",
    "higher_levels",
    "school",
    "cast_time",
    "range",
    "duration",
    "material_desc",
    "damage_type",
    "dc_type",
    "attack_type",
    "source"
]:
    df_all[col] = (
        df_all[col]
        .fillna("")
        .apply(lambda x: clean_str(str(x)) if isinstance(x, str) else "")
    )

# fix levels
df_all["level"] = (
    pd.to_numeric(df_all["level"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# remove bad rows
df_all = df_all[df_all["name"].str.strip() != ""]
df_all = df_all[df_all["desc"].str.len() > 30]

# keep longest description
df_all["_desc_len"] = df_all["desc"].str.len()

df_all = df_all.sort_values("_desc_len", ascending=False)

df_all = df_all.drop_duplicates(
    subset="name",
    keep="first"
)

df_all = df_all.drop(columns=["_desc_len"])

df_all = df_all.reset_index(drop=True)

print(f"Rows after dedup: {len(df_all)}")

print("Spells per school")
print(df_all["school"].value_counts().to_string())

print("Spells per level")
print(
    df_all["level"]
    .value_counts()
    .sort_index()
    .to_string()
)

print(f"With damage type: {(df_all['damage_type'] != '').sum()}")
print(f"With save DC: {(df_all['dc_type'] != '').sum()}")
print(f"With higher levels: {(df_all['higher_levels'].str.strip() != '').sum()}")

df_all.to_csv(OUT_CSV, index=False)

print(f"Saved -> {OUT_CSV}")

print(
    df_all[
        [
            "name",
            "school",
            "level",
            "cast_time",
            "damage_type",
            "dc_type",
            "source"
        ]
    ].head(10).to_string()
)

Loading 5etools data


 31%|███▏      | 5/16 [00:00<00:00, 28.33it/s]

Skipping spells-srd


 50%|█████     | 8/16 [00:00<00:00, 28.70it/s]

Skipping spells-erlw
Skipping spells-llok


 94%|█████████▍| 15/16 [00:00<00:00, 34.37it/s]

Skipping spells-dodk


100%|██████████| 16/16 [00:00<00:00, 25.71it/s]

Skipping spells-ghloe


5etools rows: 519
Loading Open5e data
Open5e rows: 1435
Parsed Open5e rows: 1435
Loading original CSV
Original rows: 319
Rows before dedup: 2273
Rows after dedup: 1335
Spells per school
school
Transmutation    293
Evocation        238
Conjuration      232
Enchantment      138
Necromancy       135
Abjuration       119
Divination       111
Illusion          69
Spells per level
level
0     91
1    218
2    232
3    215
4    163
5    140
6     92
7     78
8     56
9     50
With damage type: 158
With save DC: 188
With higher levels: 515
Saved -> spells_master.csv
                       name         school  level                             cast_time  damage_type dc_type    source
0            Prismatic Wall     Abjuration      9                              1 action                       original
1                    Symbol     Abjuration      7                              1 minute                       original
2                  Teleport    Conjuration      7                             